# Baseline V1 — brief client → recommandation

Moteur de reference pour la Gate 1, a battre par la V2. Le code est dans `src/moteur_v1.py`,
partage avec l'API (`api/main.py`) : le notebook et l'API donnent les memes resultats.

1. **Division** (Flavors / Fragrance) : comptage de mots-cles dans le brief.
2. **Recherche** : TF-IDF du brief contre `product_name` + `ingredients_text` + `query_category`,
   restreinte a la division detectee, similarite cosinus, top 5 avec score.

Donnees : `data/processed/sample_v1.csv` (2 193 produits, France, langue `fr`).
Aucun apprentissage : pas de donnees d'entrainement etiquetees a ce stade.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path("..").resolve()))  # racine data-brique/ pour importer src
from src import moteur_v1
from src.moteur_v1 import MoteurV1, detecter_division

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 160)

catalogue = pd.read_csv(moteur_v1.CATALOGUE_DEFAUT, dtype=str)
catalogue.groupby(["division", "query_category"]).size()

division   query_category        
flavors    boissons_aromatisees      200
           boissons_lactees          200
           boissons_proteinees       200
           boissons_vegetales        200
           desserts_aromatises       200
fragrance  deodorants                315
           gels_douche               400
           parfums                    88
           savons                    315
           savons_rasage               3
           shampoings_gels_douche     72
dtype: int64

## 1. Detection de la division

Normalisation commune (minuscules, sans accents), puis comptage des mots-cles de chaque division
(match en debut de mot : `proteine` couvre `proteinee` ; `frais` est matche en entier pour ne pas
compter dans `fraise`). La division qui a le plus de mots-cles gagne ; egalite -> `indeterminee`.

In [2]:
for division, mots in moteur_v1.MOTS_CLES.items():
    print(f"{division} ({len(mots)}) : {', '.join(mots)}")

flavors (21) : arome, aromatise, gout, saveur, boisson, proteine, yaourt, lait, dessert, sirop, soda, the glace, cafe, smoothie, creme dessert, vanille, fraise, chocolat, menthe, caramel, sportif
fragrance (19) : parfum, senteur, odeur, fragrance, savon, deodorant, gel douche, douche, shampoing, cosmetique, eau de toilette, boise, floral, musc, ambre, frais, fraiche, hygiene, peau


## 2. Recherche TF-IDF

Texte indexe : `product_name` + `ingredients_text` + `query_category` (underscores remplaces par des
espaces). Unigrammes et bigrammes, accents retires, TF sous-lineaire. Un index par division pour
que l'IDF reflete le vocabulaire propre a chaque division.

In [3]:
moteur = MoteurV1()
for division, (df_div, _, matrice) in moteur.index.items():
    print(f"{division}: {matrice.shape[0]} produits, {matrice.shape[1]} termes")

flavors: 1000 produits, 17151 termes
fragrance: 1193 produits, 18492 termes


## 3. Test sur 5 briefs

Division attendue fixee a la main pour chaque brief.

In [4]:
BRIEFS = [
    ("boisson protéinée vanille pour sportifs, base pois", "flavors"),
    ("arôme fraise pour yaourt à boire", "flavors"),
    ("senteur premium fraîche pour une gamme de savons", "fragrance"),
    ("déodorant parfum boisé pour homme", "fragrance"),
    ("sirop goût menthe pour cafés", "flavors"),
]
COLONNES = ["code", "product_name", "brands", "query_category", "format", "score"]

bilan = []
for brief, attendue in BRIEFS:
    res = moteur.recommander(brief)
    correcte = res["division"] == attendue
    bilan.append({"brief": brief, "attendue": attendue, "detectee": res["division"],
                  "mots_cles": res["scores_division"], "correcte": correcte,
                  "score_top1": res["top"]["score"].iloc[0]})
    print(f"\n=== {brief}")
    print(f"division detectee : {res['division']} {res['scores_division']} "
          f"-> attendue {attendue} : {'CORRECTE' if correcte else 'INCORRECTE'}")
    display(res["top"][COLONNES])


=== boisson protéinée vanille pour sportifs, base pois
division detectee : flavors {'flavors': 4, 'fragrance': 0} -> attendue flavors : CORRECTE


,code,product_name,brands,query_category,format,score
0,3257985312964,Boisson au soja vanille,Nature Bio,boissons_vegetales,NaN,0.103
1,10901809,Boisson Soja Vanille,"Bio Village, Marque Repère",boissons_vegetales,NaN,0.093
2,3229820806006,Barista avoine vanille,Bjorg,boissons_vegetales,NaN,0.090
3,3021760406403,Dessert Végétal Vanille de Madagascar,Materne,boissons_vegetales,NaN,0.090
4,5400706614726,Protéine De Pois Et Riz en Poudre,Purasana,boissons_proteinees,poudre,0.089



=== arôme fraise pour yaourt à boire
division detectee : flavors {'flavors': 3, 'fragrance': 0} -> attendue flavors : CORRECTE


,code,product_name,brands,query_category,format,score
0,3335030030404,Yaourt à boire fermier Fraise,le Craulois,boissons_lactees,NaN,0.213
1,3596710434619,Yaourt a boiresaveur fraise,Auchan,boissons_lactees,NaN,0.196
2,3258561420189,Yaourt à boire fraise,Belle France,boissons_lactees,NaN,0.191
3,3329770063273,Yaourt à boire sucré aromatisé,YoplaitYop,boissons_lactees,NaN,0.189
4,3350033646797,Yaourt à boire fraise,Monoprix,boissons_lactees,NaN,0.186



=== senteur premium fraîche pour une gamme de savons
division detectee : fragrance {'flavors': 0, 'fragrance': 3} -> attendue fragrance : CORRECTE


,code,product_name,brands,query_category,format,score
0,5410036269626,Senteur thé vert,"Dettol, Reckitt Benckiser",savons,NaN,0.175
1,3700214851029,Senteur Amande,"Gopack, Sans marque",savons,NaN,0.170
2,3298839232083,Eau de senteur pour bébé,Christine Arbel,parfums,NaN,0.154
3,3331840871633,Eau de toilette senteur Vanille Framboise,Les petits plaisirs,parfums,NaN,0.125
4,8712561112765,Monsavon Déodorant Femme Spray Grenade Tellement Fraîche...,Monsavon,deodorants,NaN,0.106



=== déodorant parfum boisé pour homme
division detectee : fragrance {'flavors': 0, 'fragrance': 3} -> attendue fragrance : CORRECTE


,code,product_name,brands,query_category,format,score
0,3560070917488,Déodorant parfum floral,Carrefour,deodorants,NaN,0.294
1,3560070923465,déodorant parfum musk 24h simply choice,"Carrefour, simply choice",deodorants,NaN,0.217
2,8712561456234,Déodorant Homme Spray Apollo,Axe,deodorants,NaN,0.120
3,8710847885563,AXE Ice Cool Déodorant Homme Anti Transpirant Homme Ment...,Axe,deodorants,NaN,0.110
4,8710522559383,AXE Déodorant Homme Spray Wild Lot 6X150ml,AXE,deodorants,NaN,0.103



=== sirop goût menthe pour cafés
division detectee : flavors {'flavors': 4, 'fragrance': 0} -> attendue flavors : CORRECTE


,code,product_name,brands,query_category,format,score
0,3660668209510,Sirop de Menthe,Frigolet,boissons_aromatisees,NaN,0.420
1,3092712200226,Sirop de Menthe Verte,Teisseire,boissons_aromatisees,NaN,0.399
2,3560070339228,Sirop Menthe,Simpl,boissons_aromatisees,NaN,0.319
3,5400113504276,Sirop De Menthe Bio,Delhaize,boissons_aromatisees,NaN,0.255
4,3350033184169,"Sirop à diluer à la menthe, sans sucre",Monoprix,boissons_aromatisees,NaN,0.170


In [5]:
bilan_df = pd.DataFrame(bilan)
print(f"Division correcte : {bilan_df['correcte'].sum()} / {len(bilan_df)}")
bilan_df

Division correcte : 5 / 5


,brief,attendue,detectee,mots_cles,correcte,score_top1
0,"boisson protéinée vanille pour sportifs, base pois",flavors,flavors,"{'flavors': 4, 'fragrance': 0}",True,0.103
1,arôme fraise pour yaourt à boire,flavors,flavors,"{'flavors': 3, 'fragrance': 0}",True,0.213
2,senteur premium fraîche pour une gamme de savons,fragrance,fragrance,"{'flavors': 0, 'fragrance': 3}",True,0.175
3,déodorant parfum boisé pour homme,fragrance,fragrance,"{'flavors': 0, 'fragrance': 3}",True,0.294
4,sirop goût menthe pour cafés,flavors,flavors,"{'flavors': 4, 'fragrance': 0}",True,0.420


## Lecture des resultats

| Brief | Pertinence du top 5 |
|---|---|
| boisson proteinee vanille, base pois | Faible : 4 boissons vegetales vanille, la proteine de pois seulement 5e |
| arome fraise pour yaourt a boire | Bonne : 5 yaourts a boire, dont 4 a la fraise |
| senteur premium fraiche, savons | Moyenne : 2 savons en tete, puis parfums et deodorant |
| deodorant parfum boise pour homme | Partielle : deodorants (3 pour homme), aucun boise |
| sirop gout menthe pour cafes | Tres bonne : 5 sirops de menthe |

## Limites de la baseline

- La detection de division repose sur une liste de mots-cles ecrite a la main : un brief sans aucun
  mot de la liste tombe en `indeterminee`, et un mot ambigu (`menthe`, `vanille` existent dans les
  deux divisions) peut faire basculer la decision.
- Le TF-IDF compare des mots, pas des sens : `proteinee` ne trouve pas `proteine`, `savons` ne trouve
  pas `savon`, `boise` ne trouve pas `cedre` ou `santal`, `premium` n'a pas d'equivalent.
- `ingredients_text` est long et domine le vecteur : le nom du produit et la categorie pesent peu.
- Les scores ne sont pas comparables d'une division a l'autre (index separes).
- Pas de verite terrain produit : seule la division est evaluee ici. La validation sur 30 cas de
  reference (jeudi) devra noter aussi la pertinence du top 5.

Pistes V2 : lemmatisation ou n-grammes de caracteres, ponderation des champs, vocabulaire
olfactif/aromatique (IFRA, familles de notes), embeddings de phrases multilingues, extraction
d'application et de notes.